# 12 Training Curves — Baseline vs SDD

SDD가 baseline보다 빠르게 수렴하고 더 낮은 FID를 달성하는지 epoch 단위로 추적합니다.

In [ ]:
import torch
from src.experiments import load_cfg, deep_update

# ── GPU 자동 감지 ───────────────────────────────────────────────
n_gpu = torch.cuda.device_count()
print(f"감지된 GPU 수: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory // 1024**3} GB")
if n_gpu == 0:
    print("  (GPU 없음 — CPU로 실행됩니다)")


In [ ]:
from src.experiments import load_cfg, deep_update, launch_train
from pathlib import Path
import pandas as pd

base_cfg = load_cfg("configs/cifar10.yaml")
base_cfg = deep_update(base_cfg, {
    "train": {"epochs": 200},   # 빠른 테스트; 전체 실험은 400으로 변경
    "wandb": {"enabled": True, "tags": ["cifar10", "training_curve"]},
})

variants = {
    "mse_only": {"sdd": {"enabled": False, "use_centering": False,
                         "use_sharpening": False, "use_gating": False,
                         "use_projection_head": False, "lambda_distill": 0.0}},
    "full_sdd": {"sdd": {"enabled": True,  "use_centering": True,
                         "use_sharpening": True,  "use_gating": True,
                         "use_projection_head": True}},
}

curves = {}
for name, overrides in variants.items():
    cfg = deep_update(base_cfg, overrides)
    cfg["run_name"] = f"cifar10_{name}_curve"
    print(f"\n▶ 학습 중: {name}")
    launch_train(cfg, num_processes=None, with_eval=True)
    csv = Path(f"outputs/logs/cifar10_{name}_curve_history.csv")
    if csv.exists():
        curves[name] = pd.read_csv(csv)

print("\n완료된 variant:", list(curves.keys()))


In [ ]:
import matplotlib.pyplot as plt

if curves:
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    palette = {"mse_only": "steelblue", "full_sdd": "darkorange"}

    for name, df in curves.items():
        df_eval = df.dropna(subset=["val_fid"]) if "val_fid" in df.columns else df
        c = palette.get(name, "gray")
        axes[0, 0].plot(df["epoch"], df["loss_total"], label=name, color=c)
        axes[0, 1].plot(df["epoch"], df["loss_mse"],   label=name, color=c)
        if not df_eval.empty:
            axes[1, 0].plot(df_eval["epoch"], df_eval["val_fid"],              label=name, color=c, marker="o", ms=4)
            axes[1, 1].plot(df_eval["epoch"], df_eval["val_linear_probe_acc"], label=name, color=c, marker="o", ms=4)

    for ax, title, ylabel in zip(axes.flat,
            ["Total loss", "MSE loss", "FID ↓", "Linear probe acc ↑"],
            ["loss", "loss", "FID", "accuracy"]):
        ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
        ax.legend(); ax.grid(True, alpha=0.3)

    plt.suptitle("Baseline vs SDD — training curves (CIFAR-10)", fontsize=14)
    plt.tight_layout()
    plt.savefig("outputs/figures/training_curves.png", dpi=150)
    plt.show()

    summary = {name: {
        "final_fid":       df.dropna(subset=["val_fid"])["val_fid"].iloc[-1] if "val_fid" in df.columns and df["val_fid"].notna().any() else None,
        "final_probe_acc": df.dropna(subset=["val_linear_probe_acc"])["val_linear_probe_acc"].iloc[-1] if "val_linear_probe_acc" in df.columns and df["val_linear_probe_acc"].notna().any() else None,
    } for name, df in curves.items()}
    pd.DataFrame(summary).T
